# Bird Classification with Transformers (Swin)

## 1. Setup & GPU Check

In [ ]:
# Check GPU availability
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## 2. Mount Google Drive & Install Packages

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install timm -q
print("✓ Packages installed!")

## 3. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import os
from collections import Counter
import random
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

# Torchvision imports
import torchvision.transforms as T

# timm for transformers
import timm

# Sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Progress bar
from tqdm import tqdm

print("✓ All imports successful!")

## 4. Set Paths (UPDATE THIS WITH YOUR PATHS)

In [ ]:
# ⚠️ UPDATE THIS PATH TO YOUR GOOGLE DRIVE LOCATION
BASE_PATH = Path("/content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files")

TRAIN_PATH = BASE_PATH / "train_images"
VAL_PATH = BASE_PATH / "val_images"
TEST_PATH = BASE_PATH / "test_images"

# Create output directory
OUTPUT_PATH = Path("/content/drive/MyDrive/bird_classification/outputs")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Verify paths
print(f"Base path exists: {BASE_PATH.exists()}")
print(f"Train path exists: {TRAIN_PATH.exists()}")
print(f"Val path exists: {VAL_PATH.exists()}")
print(f"Output path: {OUTPUT_PATH}")

## 5. Setup & Configuration

In [ ]:
# Set seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Configuration - Optimized for Colab T4 GPU
class Config:
    # Model
    model_name = 'swin_tiny_patch4_window7_224'  # Fast & efficient
    num_classes = 20
    
    # Transfer Learning
    use_transfer_learning = True
    freeze_backbone_epochs = 15
    unfreeze_epochs = 10
    freeze_backbone_always = False
    unfreeze_all_layers = True
    unfreeze_last_n_blocks = 2
    
    # Training
    batch_size = 32  # Good for T4 GPU
    learning_rate_head = 1e-3
    learning_rate_backbone = 1e-5
    weight_decay = 1e-4
    label_smoothing = 0.1
    
    # K-Fold
    n_folds = 3  # Faster than 5
    
    # Image
    img_size = 224
    
    # TTA
    tta_transforms = 5
    tta_voting = 'mean'
    ensemble_voting = 'mean'
    
    # Augmentation
    rotation_degrees = 10
    color_jitter = {'brightness': 0.2, 'contrast': 0.2, 'saturation': 0.2, 'hue': 0.1}
    random_erasing_prob = 0.2
    use_mixup = False
    use_cutmix = False
    
    # Regularization
    patience = 10
    dropout_rate = 0.3
    
    # Checkpoint
    checkpoint_strategy = 'best'
    save_all_epochs = False
    selected_folds = None
    
    @property
    def total_epochs(self):
        if self.freeze_backbone_always:
            return self.freeze_backbone_epochs + self.unfreeze_epochs
        else:
            return self.freeze_backbone_epochs + self.unfreeze_epochs

config = Config()
print(f"Model: {config.model_name}")
print(f"Image size: {config.img_size}")
print(f"Batch size: {config.batch_size}")
print(f"Total epochs: {config.total_epochs}")
print(f"K-Folds: {config.n_folds}")

## 6. Load Data

In [ ]:
# Load metadata
train_df = pd.read_csv(BASE_PATH / "train_metadata.csv")
val_df = pd.read_csv(BASE_PATH / "val_metadata.csv")
sample_submission = pd.read_csv(BASE_PATH / "sample_submission.csv")

print(f"Train samples: {len(train_df)}")
print(f"Val samples: {len(val_df)}")
print(f"Test samples: {len(sample_submission)}")

# Class mapping
class_mapping = {
    'Groove_billed_Ani': 0, 'Red_winged_Blackbird': 1, 'Rusty_Blackbird': 2,
    'Gray_Catbird': 3, 'Brandt_Cormorant': 4, 'Eastern_Towhee': 5,
    'Indigo_Bunting': 6, 'Brewer_Blackbird': 7, 'Painted_Bunting': 8,
    'Bobolink': 9, 'Lazuli_Bunting': 10, 'Yellow_headed_Blackbird': 11,
    'American_Crow': 12, 'Fish_Crow': 13, 'Brown_Creeper': 14,
    'Yellow_billed_Cuckoo': 15, 'Yellow_breasted_Chat': 16,
    'Black_billed_Cuckoo': 17, 'Gray_crowned_Rosy_Finch': 18,
    'Bronzed_Cowbird': 19
}

print(f"Number of classes: {len(class_mapping)}")

In [ ]:
# Prepare combined data for k-fold
train_images_df = train_df.copy()
train_images_df['full_path'] = train_images_df['path'].apply(lambda x: TRAIN_PATH / x)
train_images_df['source'] = 'train'

val_images_df = val_df.copy()
val_images_df['full_path'] = val_images_df['path'].apply(lambda x: VAL_PATH / x)
val_images_df['source'] = 'val'

combined_df = pd.concat([train_images_df, val_images_df], ignore_index=True)
combined_df['class_idx'] = combined_df['class'].map(class_mapping)

print(f"Total samples for K-Fold: {len(combined_df)}")
print(f"Samples per class: {len(combined_df) / len(class_mapping):.1f}")

labels = combined_df['class_idx'].values

## 7. Data Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_train_transforms(config):
    return T.Compose([
        T.Resize((config.img_size, config.img_size)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=config.rotation_degrees),
        T.ColorJitter(**config.color_jitter),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        T.RandomErasing(p=config.random_erasing_prob, scale=(0.02, 0.2))
    ])

def get_val_transforms(config):
    return T.Compose([
        T.Resize((config.img_size, config.img_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def get_tta_transforms(config):
    tta_list = [
        T.Compose([
            T.Resize((config.img_size, config.img_size)),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]),
        T.Compose([
            T.Resize((config.img_size, config.img_size)),
            T.RandomHorizontalFlip(p=1.0),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]),
        T.Compose([
            T.Resize(int(config.img_size * 1.1)),
            T.CenterCrop(config.img_size),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    ]
    return tta_list[:config.tta_transforms]

print("✓ Transforms defined")

## 8. Dataset Class

In [ ]:
class BirdsCombinedDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['full_path']
        label = row['class_idx']
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

print("✓ Dataset class defined")

## 9. Model Functions

In [ ]:
def create_model(config, pretrained=True, freeze_backbone=False):
    model = timm.create_model(
        config.model_name,
        pretrained=pretrained,
        num_classes=config.num_classes,
        drop_rate=config.dropout_rate
    )
    
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
        
        if hasattr(model, 'head'):
            for param in model.head.parameters():
                param.requires_grad = True
        elif hasattr(model, 'classifier'):
            for param in model.classifier.parameters():
                param.requires_grad = True
    
    return model

def unfreeze_model_backbone(model):
    for param in model.parameters():
        param.requires_grad = True
    return model

print("✓ Model functions defined")

## 10. Training Functions

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(dataloader, desc='Validation', leave=False):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

print("✓ Training functions defined")

## 11. K-Fold Training

In [ ]:
def train_kfold(config, combined_df, device):
    set_seed(42)
    
    skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)
    train_transforms = get_train_transforms(config)
    val_transforms = get_val_transforms(config)
    
    fold_results = []
    fold_models = []
    labels = combined_df['class_idx'].values
    
    print(f"Starting {config.n_folds}-Fold Cross Validation")
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(combined_df, labels)):
        print(f"\nFOLD {fold + 1}/{config.n_folds}")
        
        train_fold_df = combined_df.iloc[train_idx]
        val_fold_df = combined_df.iloc[val_idx]
        
        train_dataset = BirdsCombinedDataset(train_fold_df, transform=train_transforms)
        val_dataset = BirdsCombinedDataset(val_fold_df, transform=val_transforms)
        
        train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)
        
        model = create_model(config, pretrained=True, freeze_backbone=True).to(device)
        criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
        
        best_val_acc = 0.0
        best_model_state = None
        patience_counter = 0
        
        # Stage 1: Frozen backbone
        print(f"Stage 1: Training head ({config.freeze_backbone_epochs} epochs)")
        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = optim.AdamW(head_params, lr=config.learning_rate_head, weight_decay=config.weight_decay)
        scheduler = CosineAnnealingLR(optimizer, T_max=config.freeze_backbone_epochs, eta_min=1e-7)
        
        for epoch in range(config.freeze_backbone_epochs):
            print(f"Epoch {epoch+1}/{config.freeze_backbone_epochs}")
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device)
            val_loss, val_acc = validate(model, val_loader, criterion, device)
            
            print(f"Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
            print(f"Val: Loss={val_loss:.4f}, Acc={val_acc:.2f}%")
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model_state = model.state_dict().copy()
                patience_counter = 0
                print(f"✓ New best: {val_acc:.2f}%")
            else:
                patience_counter += 1
            
            if patience_counter >= config.patience:
                print("Early stopping")
                break
            
            scheduler.step()
        
        # Stage 2: Fine-tune (if enabled)
        if not config.freeze_backbone_always and config.unfreeze_epochs > 0:
            print(f"\nStage 2: Fine-tuning ({config.unfreeze_epochs} epochs)")
            model = unfreeze_model_backbone(model)
            
            optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate_backbone, weight_decay=config.weight_decay)
            scheduler = CosineAnnealingLR(optimizer, T_max=config.unfreeze_epochs, eta_min=1e-8)
            
            patience_counter = 0
            
            for epoch in range(config.unfreeze_epochs):
                print(f"Epoch {epoch+1}/{config.unfreeze_epochs}")
                train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device)
                val_loss, val_acc = validate(model, val_loader, criterion, device)
                
                print(f"Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
                print(f"Val: Loss={val_loss:.4f}, Acc={val_acc:.2f}%")
                
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_model_state = model.state_dict().copy()
                    patience_counter = 0
                    print(f"✓ New best: {val_acc:.2f}%")
                else:
                    patience_counter += 1
                
                if patience_counter >= config.patience:
                    print("Early stopping")
                    break
                
                scheduler.step()
        
        model.load_state_dict(best_model_state)
        fold_results.append({'fold': fold + 1, 'best_val_acc': best_val_acc})
        fold_models.append(model)
        
        print(f"Fold {fold + 1} best accuracy: {best_val_acc:.2f}%")
        
        torch.cuda.empty_cache()
    
    # Summary
    print("\n" + "="*70)
    accuracies = [r['best_val_acc'] for r in fold_results]
    for r in fold_results:
        print(f"Fold {r['fold']}: {r['best_val_acc']:.2f}%")
    print(f"Mean: {np.mean(accuracies):.2f}% ± {np.std(accuracies):.2f}%")
    print("="*70)
    
    return fold_models, fold_results

print("✓ K-Fold function defined")

## 12. RUN TRAINING

In [ ]:
# Train all folds
fold_models, fold_results = train_kfold(config, combined_df, device)

## 13. Save Models

In [ ]:
# Save models to Google Drive
for i, model in enumerate(fold_models):
    save_path = OUTPUT_PATH / f"fold_{i+1}_model.pth"
    torch.save(model.state_dict(), save_path)
    print(f"✓ Saved fold {i+1} model")

# Save results
import pickle
with open(OUTPUT_PATH / 'fold_results.pkl', 'wb') as f:
    pickle.dump(fold_results, f)

print(f"\n✓ All models and results saved to {OUTPUT_PATH}")

## 14. Test Set Predictions (Optional)

In [ ]:
# Make predictions on test set
@torch.no_grad()
def predict_ensemble(models, test_loader, device):
    all_preds = []
    
    for model in models:
        model.eval()
        preds = []
        
        for images, _ in tqdm(test_loader, desc='Predicting'):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds.append(probs.cpu().numpy())
        
        all_preds.append(np.vstack(preds))
    
    # Average predictions
    ensemble_preds = np.mean(all_preds, axis=0)
    final_preds = ensemble_preds.argmax(axis=1)
    
    return final_preds

# Create test dataset
class TestDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.image_files = sorted(list(self.image_dir.glob('*/*.jpg')))
        self.transform = transform
        
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        return image, img_path.name

test_dataset = TestDataset(TEST_PATH, transform=get_val_transforms(config))
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)

predictions = predict_ensemble(fold_models, test_loader, device)

# Create submission
submission = sample_submission.copy()
submission['class'] = predictions
submission.to_csv(OUTPUT_PATH / 'submission.csv', index=False)

print("✓ Submission saved!")